# 챗봇 평가 하네스 합성 점검
DB/LLM을 호출하지 않고 질문 생성 로직과 지표 계산 흐름을 확인합니다. 출력은 저장하지 마세요.

In [ ]:
from pathlib import Path
import csv, json, sys, tempfile
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())  # 노트북 위치와 무관하게 저장소 루트를 찾는다
repo = REPO_ROOT
chat_dir = repo / 'src/card_benefit/payment_card/file_pipeline/chat'
sys.path.insert(0, str(chat_dir))
import eval_questions, run_eval
synthetic_csv_dir = Path(tempfile.mkdtemp(prefix='fp-chat-check-'))
def write_csv(name, cols, rows):
    with (synthetic_csv_dir / name).open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)
write_csv('fp_nodes.csv', ['id','node_type','props'], [
 {'id':'c1','node_type':'Card','props':json.dumps({'product_name':'가나다 카드'})},
 {'id':'b1','node_type':'Benefit','props':json.dumps({'name':'커피 할인'})},
 {'id':'l1','node_type':'Limit','props':json.dumps({'period':'월','amount':20000})},
 {'id':'k1','node_type':'Condition','props':json.dumps({'text':'전월 30만원'})},
 {'id':'t1','node_type':'Target','props':json.dumps({'text':'해외 가맹점'})},
 {'id':'x1','node_type':'Target','props':json.dumps({'text':'상품권 구매'})}])
write_csv('fp_edges.csv', ['id','source_id','target_id','edge_type'], [
 {'id':'e1','source_id':'c1','target_id':'b1','edge_type':'OFFERS'}, {'id':'e2','source_id':'b1','target_id':'l1','edge_type':'HAS_LIMIT'},
 {'id':'e3','source_id':'b1','target_id':'k1','edge_type':'HAS_CONDITION'}, {'id':'e4','source_id':'b1','target_id':'t1','edge_type':'APPLIES_TO'}, {'id':'e5','source_id':'b1','target_id':'x1','edge_type':'EXCLUDES'}])
write_csv('fp_evidence.csv', ['node_id','edge_id','doc_sha256','quote_status'], [{'node_id':'b1','edge_id':'','doc_sha256':'a'*64,'quote_status':'located'}])
questions, skipped = eval_questions.generate([synthetic_csv_dir], per_type=2, seed=7)
questions[:2], skipped


In [ ]:
synthetic_records = [{
  'question': {'expected': ['월', '20000'], 'expected_node_names': ['커피 할인']},
  'result': {'answer': '월 한도 20000원', 'sources': [{'title': '문서 12345678'}], 'subgraph': {'nodes': [{'name': '커피 할인'}]}},
  'elapsed_seconds': 0.25
}]
run_eval.metrics(synthetic_records)


## LLM 없는 검색 평가
합성 그래프에서 seed 선택, hop 탐색, 기대값 재현율과 크기를 계산합니다.

In [ ]:
from pathlib import Path
import sys
chat_dir = repo / 'src/card_benefit/payment_card/file_pipeline/chat'
if str(chat_dir) not in sys.path: sys.path.insert(0, str(chat_dir))
import retrieval_eval
synthetic_nodes = [
 {'id':'c','name':'가나다 카드','label':'Card','props':{}},
 {'id':'b','name':'커피 할인','label':'Benefit','props':{}},
 {'id':'l','name':'월 2만원','label':'Limit','props':{'amount':20000}},
 {'id':'d','name':'문서 abcdef12','label':'Document','props':{'summary':'synthetic'}}]
synthetic_edges = [{'source':'c','target':'b','type':'OFFERS'}, {'source':'b','target':'l','type':'HAS_LIMIT'}, {'source':'b','target':'d','type':'SUPPORTED_BY'}]
synthetic_question = {'id':'demo','type':'card_benefits','question':'가나다 카드 혜택','seed_entities':['가나다 카드'],'expected':['커피 할인'],'expected_node_names':['가나다 카드','커피 할인']}
retrieval_eval.evaluate([synthetic_question], synthetic_nodes, synthetic_edges, seed_mode='contains', hops=2)


In [ ]:
# 합성 노드로 조사 제거와 fuzzy 시드 선택을 점검합니다.
from pathlib import Path
import sys
CHAT_DIR = Path.cwd() / "src/card_benefit/payment_card/file_pipeline/chat"
if not CHAT_DIR.exists():
    CHAT_DIR = Path.cwd()
    while CHAT_DIR != CHAT_DIR.parent and not (CHAT_DIR / "seed_matcher.py").exists():
        CHAT_DIR = CHAT_DIR.parent
sys.path.insert(0, str(CHAT_DIR))
from seed_matcher import select_seeds_fuzzy
synthetic_nodes = [
    {"name": "합성 카드 혜택", "label": "Benefit"},
    {"name": "합성 카드", "label": "Card"},
    {"name": "합성 조건", "label": "Condition"},
]
select_seeds_fuzzy(synthetic_nodes, "합성 카드의 혜택은?")


## 합성 답변 판정 규칙
기대값 정규화, 거절 패턴, 검색 성공·답변 실패와 검색 실패 분류를 합성 자료로 확인합니다.

In [ ]:
from pathlib import Path
import sys
chat_dir = repo / "src/card_benefit/payment_card/file_pipeline/chat"
if str(chat_dir) not in sys.path: sys.path.insert(0, str(chat_dir))
from answer import answer_eval
synthetic_question = {"expected": ["₩20,000", "해외 가맹점"], "seed_entities": ["합성 카드"]}
synthetic_result = {"answer": "월 20000원", "sources": [], "subgraph": {"nodes": [{"name": "합성 카드", "props": {"limit": 20000, "target": "해외 가맹점"}}]}}
judgement = answer_eval.judge_answer(synthetic_question, synthetic_result, ["합성 카드"])
assert judgement["expected_values_in_answer"] == 1
assert judgement["refusal"] is False
assert judgement["classification"] == "search_hit_answer_miss"
synthetic_refusal = answer_eval.judge_answer({"expected": ["혜택"], "seed_entities": []}, {"answer": "관련 정보가 없습니다", "sources": [], "subgraph": {"nodes": []}}, [])
assert synthetic_refusal["refusal"] is True and synthetic_refusal["classification"] == "search_miss"
judgement, synthetic_refusal


## 골든셋 변환과 검수 시트 합성 점검
상단 경로 변수를 바꾸어 합성 CSV/JSONL로 두 도구를 확인합니다. 실제 답변·문서 원문은 노트북에 저장하지 않습니다.

In [ ]:
from pathlib import Path
import csv, json, tempfile, sys
chat_dir = repo / "src/card_benefit/payment_card/file_pipeline/chat"
if str(chat_dir) not in sys.path: sys.path.insert(0, str(chat_dir))
import golden_set, review_sheet
check_dir = Path(tempfile.mkdtemp(prefix="fp-golden-check-"))
csv_path = check_dir / "synthetic.csv"
with csv_path.open("w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=golden_set.HEADERS); w.writeheader()
    w.writerow(dict(zip(golden_set.HEADERS, ["SYN-1","하나","혜택조회","가짜 카드 혜택?","가짜 혜택","","","synthetic.pdf",1,"아니오","아니오","둘다","작성자","검수자","합성","가짜 카드"])))
questions = golden_set.convert(csv_path)
question_json = check_dir / "questions.json"; question_json.write_text(json.dumps({"questions": questions}, ensure_ascii=False), encoding="utf-8")
result_jsonl = check_dir / "results.jsonl"; result_jsonl.write_text(json.dumps({"id":"SYN-1","answer":"테스트 답변","retrieved_node_ids":[],"selected_seed_names":[],"classification":"search_miss"}, ensure_ascii=False)+"\n", encoding="utf-8")
for name, cols in [("fp_nodes.csv", ["id","node_type","props"]),("fp_edges.csv", ["id","source_id","target_id","edge_type"]),("fp_evidence.csv", ["node_id","edge_id","doc_sha256","page_no","start_pos","end_pos","quote_status"])]:
    with (check_dir/name).open("w", newline="", encoding="utf-8") as f: csv.DictWriter(f, fieldnames=cols).writeheader()
documents = check_dir / "documents.csv"
with documents.open("w", newline="", encoding="utf-8") as f: csv.DictWriter(f, fieldnames=["sha256","file_path"]).writeheader()
review_sheet.make_sheet(result_jsonl, question_json, [check_dir], [documents], check_dir/"review.html")


## 골든셋 페이지 샘플러 (합성 PDF)
상단 경로 변수만 바꿔 합성 인덱스와 PDF를 만든 뒤 실행합니다. PDF 원문은 저장하지 않습니다.


In [ ]:
from pathlib import Path
import csv, tempfile
import fitz
from payment_card.file_pipeline.chat import golden_sampler

# 경로는 점검용 임시 디렉터리 아래에 둡니다.
sample_work_dir = Path(tempfile.mkdtemp(prefix='golden-sampler-check-'))
sample_pdf_path = sample_work_dir / '합성(안내).pdf'
sample_index_path = sample_work_dir / 'documents.csv'
sample_output_path = sample_work_dir / 'draft.csv'


In [ ]:
doc = fitz.open()
for label in ('synthetic alpha ' * 20, 'synthetic beta ' * 20):
    page = doc.new_page()
    page.insert_text((72, 72), label)
doc.save(sample_pdf_path)
doc.close()
index_fields = ['issuer','doc_id','sha256','file_path','card_type','doc_role','canonical','pages','text_chars','ocr_needed']
with sample_index_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=index_fields)
    writer.writeheader()
    writer.writerow({'issuer':'hana','doc_id':'synthetic','sha256':'synthetic-hash','file_path':str(sample_pdf_path),'card_type':'sample','doc_role':'product_guide','canonical':'Y','pages':'2','text_chars':'100','ocr_needed':'N'})
sample_rows, sample_summary = golden_sampler.sample({'hana': sample_index_path}, count=2, seed=7, min_chars=50)
golden_sampler.write_csv(sample_output_path, sample_rows)
print(sample_summary)
print({'sampled_pages': sum(r['정답없음'] == '아니오' for r in sample_rows), 'slots': sum(r['정답없음'] == '예' for r in sample_rows), 'output': str(sample_output_path)})


## 숫자·이름 답변 재판정 (합성 입력)
단위 환산, 핵심 토큰, 엄격 비교와 새 판정의 근거 구간을 확인합니다.

In [ ]:
import sys
chat_dir = repo / "src/card_benefit/payment_card/file_pipeline/chat"
if str(chat_dir) not in sys.path: sys.path.insert(0, str(chat_dir))
from answer import answer_eval
synthetic_pairs = [("300000", "30만원"), ("1000000", "1,000,000원"), ("카타르항공 할인", "카타르항공 최대 15% 할인"), ("간편결제(네이버페이)", "네이버페이 간편결제")]
synthetic_matches = [answer_eval._value_match(answer, expected, "benefit_limit") for expected, answer in synthetic_pairs]
assert all(row[0] for row in synthetic_matches)
synthetic_judgement = answer_eval.judge_answer({"type":"benefit_limit", "expected":["300000", "카타르항공 할인"]}, {"answer":"30만원 이상. 카타르항공 최대 15% 할인", "subgraph":{"nodes":[]}}, [])
assert synthetic_judgement["expected_values_in_answer"] == 2 and synthetic_judgement["expected_values_in_answer_strict"] == 0
synthetic_judgement["value_matches"]


## 집중 검수 시트 합성 점검
합성 PDF 한 쪽과 합성 그래프로 정답 문장이 든 근거가 위로 오는지, 줄 번호·형광 구간·그래프 경로가 나오는지 확인합니다. 출력은 임시 폴더에만 만들고 실제 카드 문서는 쓰지 않습니다.

In [ ]:
from pathlib import Path
import csv, json, sys, tempfile
import fitz                                        # PyMuPDF: 합성 PDF 만들기
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path: sys.path.insert(0, str(REPO_ROOT / 'src'))
from card_benefit.payment_card.file_pipeline.chat import review_focus
from card_benefit.payment_card.file_pipeline.chat.review.review_sheet import uuidish

SENTENCE = "alpha payments only are provided"          # 입력 예시: 정답 문장 (합성)
work = Path(tempfile.mkdtemp(prefix="fp-focus-check-"))   # /tmp 아래여야 출력이 허용된다
pdf_path = work / "synthetic.pdf"
doc = fitz.open(); doc.new_page().insert_text((72, 72), "HEADER LINE\n" + SENTENCE + " today\nTAIL LINE"); doc.save(pdf_path); doc.close()
with fitz.open(pdf_path) as d: text = d[0].get_text()
a, b = text.index(SENTENCE), text.index("HEADER")
issuer_dir = work / "fp-check-demo"; issuer_dir.mkdir(); index_dir = work / "demo"; index_dir.mkdir()
def write_csv(path, header, rows):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as f: csv.writer(f).writerows([header, *rows])
write_csv(index_dir / "documents.csv", ["sha256", "file_path"], [["sha1", str(pdf_path)]])
write_csv(issuer_dir / "fp_nodes.csv", ["id", "node_type", "props", "chunk_id", "run_id"], [
    ["card", "Card", json.dumps({"name": "합성카드"}, ensure_ascii=False), "", "r"],
    ["ben1", "Benefit", json.dumps({"name": "정답혜택"}, ensure_ascii=False), "", "r"],
    ["ben2", "Benefit", json.dumps({"name": "딴혜택"}, ensure_ascii=False), "", "r"]])
write_csv(issuer_dir / "fp_edges.csv", ["id", "source_id", "target_id", "edge_type", "run_id"],
          [["e1", "card", "ben1", "OFFERS", "r"], ["e2", "card", "ben2", "OFFERS", "r"]])
write_csv(issuer_dir / "fp_evidence.csv", ["node_id", "edge_id", "doc_sha256", "page_no", "start_pos", "end_pos", "quote_status"], [
    ["ben2", "", "sha1", "1", str(b + 1), str(b + 6), "located"],
    ["ben1", "", "sha1", "1", str(a + 1), str(a + len(SENTENCE)), "located"]])
(work / "q.json").write_text(json.dumps({"questions": [{"id": "SYN-1", "issuer": "demo", "type": "benefit_condition", "question": "합성 질문", "expected": [SENTENCE]}]}, ensure_ascii=False), encoding="utf-8")
(work / "r.jsonl").write_text(json.dumps({"id": "SYN-1", "answer": "합성 답변", "classification": "search_hit_answer_miss", "selected_seed_names": ["합성카드"], "retrieved_node_ids": [uuidish(i) for i in ("card", "ben1", "ben2")]}, ensure_ascii=False) + "\n", encoding="utf-8")

out = review_focus.make_focus_sheet(work / "r.jsonl", work / "q.json", [issuer_dir], [index_dir / "documents.csv"], work / "focus.html")
html_text = out.read_text(encoding="utf-8")
print("★ 정답 문장 포함 표시:", "★ 정답 문장 포함" in html_text)
print("형광 구간이 정답 문장:", f"<mark>{SENTENCE}</mark>" in html_text)
print("줄 번호 표시(2줄):", "2줄" in html_text)
print("그래프 경로 표시:", "—OFFERS→ 정답혜택(Benefit)" in html_text)
print("정답 근거가 앞:", html_text.index("정답혜택") < html_text.index("딴혜택"))

## entity_dup_probe: 합성 검색 서브그래프 병합

In [ ]:
from payment_card.file_pipeline.chat.eval.entity_dup_probe import evaluate_question
from payment_card.file_pipeline.chat.eval.retrieval_eval import uuidish
class FixedMatcher:
    def select(self, question, n=3, threshold=0.15): return ['Synthetic Card']
card_id, member_a, member_b = (uuidish(value) for value in ('synthetic-card', 'synthetic-a', 'synthetic-b'))
probe_nodes = [{'id': card_id, 'name': 'Synthetic Card', 'label': 'Card', 'props': {}},
               {'id': member_a, 'name': 'Synthetic benefit', 'label': 'Benefit', 'props': {}},
               {'id': member_b, 'name': 'Synthetic benefit', 'label': 'Benefit', 'props': {}}]
probe_edges = [{'source': card_id, 'target': member_a, 'type': 'OFFERS', '_fp_id': 'edge-a'},
               {'source': card_id, 'target': member_b, 'type': 'OFFERS', '_fp_id': 'edge-b'}]
probe_evidence = [{'node_id': 'synthetic-a', 'edge_id': '', 'doc_sha256': 'sha-a', 'page_no': '1'},
                  {'node_id': 'synthetic-b', 'edge_id': '', 'doc_sha256': 'sha-b', 'page_no': '1'}]
probe_result = evaluate_question({'id': 'SYN-PROBE', 'question': 'Synthetic Card', 'evidence_page': 1},
    probe_nodes, probe_edges, probe_evidence, FixedMatcher(), {member_b: member_a}, {'group': {member_a, member_b}})
{key: probe_result[key] for key in ('node_count_before', 'node_count_after', 'evidence_count_before', 'evidence_count_after')}

## chat/eval/value_conflicts: 합성 혜택 수치 충돌 점검

정규화된 혜택명이 같은 rate/amount 값 중, 정규화 후 서로 다른 값이 서로 다른 알려진 페이지에 있을 때만 충돌로 봅니다. 같은 페이지의 티어 값과 페이지 미상 값은 충돌 판정에서 제외합니다. 카드 범위는 Card -OFFERS-> Benefit 연결로 정합니다.


In [ ]:
from payment_card.file_pipeline.chat.eval.value_conflicts import detect_value_conflicts

synthetic_subgraph = {
    "nodes": [
        {"id": "card", "name": "Synthetic Card", "label": "Card", "props": {}},
        {"id": "benefit-a", "name": "Synthetic Hotel Room 15 할인", "label": "Benefit", "props": {"rate": "15%", "source_text": "(p.31) synthetic"}},
        {"id": "benefit-b", "name": "synthetic hotel room 할인", "label": "Benefit", "props": {"rate": "10%", "source_text": "(p.47) synthetic"}},
    ],
    "edges": [
        {"source": "card", "target": "benefit-a", "type": "OFFERS"},
        {"source": "card", "target": "benefit-b", "type": "OFFERS"},
    ],
}

conflicts = detect_value_conflicts(synthetic_subgraph)
assert len(conflicts) == 1
assert {item['page'] for item in conflicts[0]['values']} == {31, 47}
detect_value_conflicts({**synthetic_subgraph, 'nodes': [
    *synthetic_subgraph['nodes'][:1],
    {**synthetic_subgraph['nodes'][1], 'props': {'rate': '1.0%', 'source_text': '(p.2) synthetic'}},
    {**synthetic_subgraph['nodes'][2], 'props': {'rate': '1%', 'source_text': '(p.3) synthetic'}},
]}) == []


## Retrieval stability 합성 점검
합성 입력으로 LLM 선택 및 고정 엔티티 반복 검색 기록과 안정성 집계를 표시합니다. DB/LLM 호출은 하지 않습니다.


In [ ]:
from pathlib import Path
import json, sys, tempfile
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())
PYTHON_ROOTS = [str(REPO_ROOT / 'src'), str(REPO_ROOT / 'src/card_benefit')]
for root in PYTHON_ROOTS:
    if root not in sys.path: sys.path.insert(0, root)
from payment_card.file_pipeline.chat import retrieval_stability
synthetic_question = {'id': 'demo-1', 'question': 'synthetic hotel benefit question'}
class FakeRag:
    def list_entities(self, question): return [{"name": "Synthetic Hotel", "label": "Benefit"}]
    def analyze(self, question, entities): return type("Analysis", (), {"entities": ["Synthetic Hotel"]})()
    def explore(self, analysis): return {"nodes": [{"id": "benefit-1", "name": "Synthetic stay", "label": "Benefit", "props": {"rate": "15%", "page_no": 47}}]}
class FrozenFakeRag:
    def analyze(self, *args): raise AssertionError("frozen mode must skip analyze")
    def explore(self, analysis): return {"nodes": [{"id": "benefit-1", "name": "Synthetic stay", "label": "Benefit", "props": {"rate": "15%", "page_no": 47}}]}
with tempfile.TemporaryDirectory(prefix='retrieval-stability-', dir='/tmp') as temp_dir:
    sample_path = Path(temp_dir) / 'records.jsonl'
    frozen_path = Path(temp_dir) / 'frozen-records.jsonl'
    retrieval_stability.run_repeated([synthetic_question], 2, FakeRag(), sample_path)
    sample_records = [json.loads(line) for line in sample_path.read_text(encoding='utf-8').splitlines()]
    retrieval_stability.run_repeated([synthetic_question], 2, FrozenFakeRag(), frozen_path, {'0': ['Synthetic Hotel']})
    frozen_records = [json.loads(line) for line in frozen_path.read_text(encoding='utf-8').splitlines()]
    sample_records, retrieval_stability.aggregate_records(sample_records), frozen_records, retrieval_stability.aggregate_records(frozen_records)


## 합성 결정적 시드 합집합
질문과 후보 목록만 사용해 Card 및 Benefit 시드 추가를 확인합니다.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())
PYTHON_ROOTS = [str(REPO_ROOT / 'src'), str(REPO_ROOT / 'src/card_benefit')]
for root in PYTHON_ROOTS:
    if root not in sys.path: sys.path.insert(0, root)
from payment_card.file_pipeline.chat.retrieval.seed_union import union_seed_names
synthetic_candidates = [
    {'name': '합성 카드', 'label': 'Card'},
    {'name': '편의점 합성 할인', 'label': 'Benefit'},
    {'name': '합성 조건', 'label': 'Condition'},
]
union_seed_names('합성 카드의 편의점 혜택', synthetic_candidates, ['LLM 선택'])


In [ ]:
from pathlib import Path
import sys
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/card_benefit").exists()), Path.cwd())
chat_dir = repo / "src/card_benefit/payment_card/file_pipeline/chat"
if str(chat_dir) not in sys.path: sys.path.insert(0, str(chat_dir))
from retrieval.source_label import label_for_document, annotate_subgraph
synthetic_paths = [
    "woori/common/국제브랜드서비스안내장_VISA-P.pdf",
    "hana/common/공통_신용카드약관.pdf",
    "cards/synthetic_card.pdf",
]
source_label_examples = [(path, label_for_document(path, "synthetic")) for path in synthetic_paths]
synthetic_graph = {"nodes": [{"id": "benefit-1", "label": "Benefit", "props": {"rate": "5%"}}], "edges": []}
labeled_graph = annotate_subgraph(synthetic_graph, {"benefit-1": ["sha-1"]}, {"sha-1": "국제브랜드 공통 안내장(VISA)"})
display({"filename_labels": source_label_examples, "annotated_subgraph": labeled_graph})
